# Package 4 — Super customer score

Target: a new label we engineer ourselves, `super_customer` (binary: 0/1) — not an
existing column in the data.

Definition (business rule, chosen with the user): a purchaser (`purchased == 1`) who
was **referred** (`referred == "Yes"`), **upsold** (`upsell == 1`), and stayed in the
**top third** of tenure among purchasers (`ltv_months >= 32`).

Model: CatBoost classifier, with a hand-engineered categorical `budget_tier` feature
(Low/Mid/High from `ad_budget`), plus hyperparameter tuning over learning rate, depth,
and iterations. Output: a 0-100 score = predicted probability * 100.

## Building the label

Non-purchasers (`purchased == 0`) can never be a super customer — they never bought
anything, so `referred`/`upsell`/`ltv_months` don't even apply to them the same way.
The label is 1 only when all three conditions hold at once.

In [1]:
import pandas as pd

df = pd.read_csv("../data/funnel_marketing_data.csv")

tenure_threshold = df.loc[df["purchased"] == 1, "ltv_months"].quantile(2/3)

df["super_customer"] = (
    (df["purchased"] == 1)
    & (df["referred"] == "Yes")
    & (df["upsell"] == 1)
    & (df["ltv_months"] >= tenure_threshold)
).astype(int)

print(f"tenure threshold (top third of purchasers): {tenure_threshold} months")
print(df["super_customer"].value_counts())
print(f"share positive: {df['super_customer'].mean()*100:.1f}%")

tenure threshold (top third of purchasers): 32.0 months
super_customer
0    2860
1     640
Name: count, dtype: int64
share positive: 18.3%


## Feature selection — early-funnel only

This package has a stricter leakage rule than Packages 2/3. The brief asks for a score
computed from a **new customer's early funnel data** — i.e. at prediction time we are
standing at the *start* of the funnel, before the customer has necessarily even
purchased. So every column that only exists *after* the funnel plays out is excluded —
and that happens to be every column used to build the label itself:

Excluded (post-outcome, unknown at prediction time): `purchased`, `not_closed`,
`closed`, `calls_to_closed`, `calls_to_not_closed`, `customer_acquisition_cost`,
`ltv_months`, `upsell`, `cumulative_profit`, `referred`.

Kept (known from the moment leads start coming in): `ad_budget`, `num_leads`,
`leads_answered`, `leads_not_answered`, `followup_1`..`followup_5`, plus the new
`budget_tier` engineered below.

In [2]:
base_features = [
    "ad_budget", "num_leads", "leads_answered", "leads_not_answered",
    "followup_1", "followup_2", "followup_3", "followup_4", "followup_5",
]

target = "super_customer"

df[base_features].describe()

,ad_budget,num_leads,leads_answered,leads_not_answered,followup_1,followup_2,followup_3,followup_4,followup_5
count,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000,3500.000000
mean,4655.342857,46.220571,27.978571,18.242000,21.895714,16.274286,13.244857,11.871143,8.401429
std,4107.864396,24.433354,14.482454,10.896667,11.628637,8.741021,7.181698,6.459602,4.692517
min,500.000000,11.000000,4.000000,5.000000,3.000000,2.000000,1.000000,1.000000,1.000000
25%,2000.000000,29.000000,16.000000,11.000000,12.000000,9.000000,7.000000,7.000000,5.000000
50%,3000.000000,40.000000,27.000000,14.000000,21.000000,15.000000,13.000000,11.000000,8.000000
75%,6000.000000,59.000000,37.000000,22.000000,29.000000,22.000000,18.000000,16.000000,11.000000
max,20000.000000,139.000000,84.000000,68.000000,66.000000,51.000000,42.000000,37.000000,27.000000


## Engineering `budget_tier`

A categorical version of `ad_budget`, split into three equal-sized buckets (tertiles)
so CatBoost can pick up non-linear budget effects (e.g. "Mid budget converts better
than High") that a single numeric column might smooth over. CatBoost handles
categorical columns natively — no one-hot encoding needed, just pass the column name
via `cat_features`.

In [3]:
df["budget_tier"] = pd.qcut(
    df["ad_budget"], q=3, labels=["Low", "Mid", "High"]
)

print(df["budget_tier"].value_counts())
df.groupby("budget_tier", observed=True)["super_customer"].mean()

budget_tier
Mid     1307
Low     1190
High    1003
Name: count, dtype: int64


budget_tier
Low     0.119328
Mid     0.381025
High    0.000000
Name: super_customer, dtype: float64

## Hyperparameter tuning

Instead of hand-picking `learning_rate`/`depth`/`iterations`, we search over them
automatically with `RandomizedSearchCV`: it tries random combinations from the ranges
below, scores each with 5-fold stratified CV on ROC-AUC (a threshold-independent metric
— appropriate since the whole point of this model is to output a *ranked probability*,
not a hard yes/no), and keeps the best-scoring combination.

`budget_tier` is passed to CatBoost via `cat_features` so it's handled natively as a
category, not manually encoded.

In [4]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from catboost import CatBoostClassifier

features = base_features + ["budget_tier"]
X = df[features].copy()
X["budget_tier"] = X["budget_tier"].astype(str)
y = df[target]

cat_feature_idx = [X.columns.get_loc("budget_tier")]

param_distributions = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15, 0.2],
    "depth": [3, 4, 5, 6, 8],
    "iterations": [100, 200, 300, 500],
}

# cat_features is passed via .fit(), not the constructor: CatBoost's sklearn wrapper
# does not clone cleanly when cat_features is a constructor arg (RandomizedSearchCV
# clones the estimator internally for each CV fold).
base_model = CatBoostClassifier(
    random_state=42,
    verbose=False,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    base_model,
    param_distributions=param_distributions,
    n_iter=20,
    scoring="roc_auc",
    cv=skf,
    random_state=42,
    n_jobs=-1,
)

search.fit(X, y, cat_features=cat_feature_idx)

print("Best params:", search.best_params_)
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

Best params: {'learning_rate': 0.01, 'iterations': 100, 'depth': 3}
Best CV ROC-AUC: 0.8132


## Full metrics with the tuned model

The search above optimized only ROC-AUC. Now run the same 5-fold CV with the winning
hyperparameters and report the full metric set (accuracy, precision, recall, F1,
ROC-AUC), the same way Package 3 did — so the numbers are comparable across
packages.

In [5]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

best_params = search.best_params_
metrics = {"accuracy": [], "precision": [], "recall": [], "f1": [], "roc_auc": []}

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        cat_features=cat_feature_idx,
        random_state=42,
        verbose=False,
        **best_params,
    )
    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    proba = model.predict_proba(X_val)[:, 1]

    metrics["accuracy"].append(accuracy_score(y_val, pred))
    metrics["precision"].append(precision_score(y_val, pred))
    metrics["recall"].append(recall_score(y_val, pred))
    metrics["f1"].append(f1_score(y_val, pred))
    metrics["roc_auc"].append(roc_auc_score(y_val, proba))

for name, values in metrics.items():
    print(f"{name}: {np.mean(values):.4f} (+/- {np.std(values):.4f})")

C:\Users\User\anaconda3\envs\funneliq\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


C:\Users\User\anaconda3\envs\funneliq\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


C:\Users\User\anaconda3\envs\funneliq\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


C:\Users\User\anaconda3\envs\funneliq\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


accuracy: 0.8171 (+/- 0.0000)
precision: 0.0000 (+/- 0.0000)
recall: 0.0000 (+/- 0.0000)
f1: 0.0000 (+/- 0.0000)
roc_auc: 0.8132 (+/- 0.0084)


C:\Users\User\anaconda3\envs\funneliq\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


## Why the 0.5 threshold doesn'''t matter here

At the default 0.5 cutoff the tuned model predicts every customer as "not super" —
precision/recall/F1 are all 0 (see above). That looks alarming but is actually fine:
we searched for the hyperparameters that maximize **ROC-AUC**, a *ranking* metric, and
the product we're building is a 0-100 **score to rank customers by**, not a hard
yes/no gate. What matters is whether high-scored customers are actually more likely to
be super customers than low-scored ones — a decile check on one validation fold makes
that concrete.

In [6]:
train_idx, val_idx = next(skf.split(X, y))
check_model = CatBoostClassifier(random_state=42, verbose=False, **best_params)
check_model.fit(X.iloc[train_idx], y.iloc[train_idx], cat_features=cat_feature_idx)

val_proba = check_model.predict_proba(X.iloc[val_idx])[:, 1]
val_y = y.iloc[val_idx].values

order = np.argsort(-val_proba)
val_y_sorted = val_y[order]
n = len(val_y_sorted)

top_decile_rate = val_y_sorted[: n // 10].mean()
bottom_half_rate = val_y_sorted[n // 2 :].mean()
base_rate = val_y.mean()

print(f"base rate (whole validation fold): {base_rate*100:.1f}%")
print(f"top decile by score: {top_decile_rate*100:.1f}% ({top_decile_rate/base_rate:.1f}x lift)")
print(f"bottom half by score: {bottom_half_rate*100:.1f}%")

base rate (whole validation fold): 18.3%
top decile by score: 42.9% (2.3x lift)
bottom half by score: 0.3%


## Final model and the 0-100 score

Train once more on the *full* dataset (no held-out fold — this is the model that would
actually ship) with the tuned hyperparameters, then convert each customer'''s predicted
probability into a 0-100 score.

In [7]:
final_model = CatBoostClassifier(
    random_state=42,
    verbose=False,
    **best_params,
)
final_model.fit(X, y, cat_features=cat_feature_idx)

df["super_customer_score"] = (final_model.predict_proba(X)[:, 1] * 100).round(1)

df[["super_customer", "super_customer_score"]].groupby("super_customer").describe()["super_customer_score"]

,count,mean,std,min,25%,50%,75%,max
super_customer,,,,,,,,
0,2860.0,27.912622,9.626831,19.0,19.7,21.8,40.2,41.5
1,640.0,40.092031,2.182398,22.2,39.7,41.0,41.4,41.5


## Business questions

**Profile of actual super customers** (, , top-third tenure —
the same rule used for the label): what share of total profit do they generate, and
what'''s their average acquisition cost, compared to everyone else?

In [8]:
profit_share = df.loc[df["super_customer"] == 1, "cumulative_profit"].sum() / df["cumulative_profit"].sum()
count_share = df["super_customer"].mean()

cac_super = df.loc[df["super_customer"] == 1, "customer_acquisition_cost"].mean()
cac_other = df.loc[df["super_customer"] == 0, "customer_acquisition_cost"].mean()

print(f"super customers are {count_share*100:.1f}% of all rows")
print(f"...but generate {profit_share*100:.1f}% of total cumulative profit")
print()
print(f"avg acquisition cost, super customers:  {cac_super:,.0f}")
print(f"avg acquisition cost, everyone else:     {cac_other:,.0f}")
print(f"ratio: {cac_super/cac_other:.2f}x")

super customers are 18.3% of all rows
...but generate 39.6% of total cumulative profit

avg acquisition cost, super customers:  1,000
avg acquisition cost, everyone else:     1,452
ratio: 0.69x


**How could Northbound spot them earlier?** Two angles: which early-funnel features
the model actually leans on (feature importance), and whether eventual super customers
already look different at the *very first* funnel signals — before any purchase
decision.

In [9]:
importance = pd.Series(
    final_model.get_feature_importance(), index=features
).sort_values(ascending=False)

print("Feature importance (final model):")
print(importance)
print()

print("Early-funnel engagement, super customers vs. others (mean values):")
compare_cols = ["ad_budget", "num_leads", "leads_answered", "leads_not_answered"] + [f"followup_{i}" for i in range(1, 6)]
print(df.groupby("super_customer")[compare_cols].mean().T)

Feature importance (final model):
ad_budget             51.701311
num_leads             12.899090
leads_not_answered    11.192292
leads_answered         5.789087
followup_2             4.649140
followup_4             4.461397
followup_3             4.133419
followup_1             3.708951
followup_5             1.465312
budget_tier            0.000000
dtype: float64

Early-funnel engagement, super customers vs. others (mean values):
super_customer                0            1
ad_budget           4974.545455  3228.906250
num_leads             47.676224    39.715625
leads_answered        28.445804    25.890625
leads_not_answered    19.230420    13.825000
followup_1            22.279371    20.181250
followup_2            16.560140    14.996875
followup_3            13.471678    12.231250
followup_4            12.073077    10.968750
followup_5             8.544755     7.760937


## Summary

- **Score separates the groups well:** super customers score a tight 40.1 +/- 2.2
  (out of 100), non-super customers 27.9 +/- 9.6, with the two distributions
  overlapping only at the very top. Confirmed by the earlier decile check: top-decile
  customers by score are 18.3% super customers in the raw data but 42.9% in the top
  decile (2.3x lift), bottom half ~0%.
- **18.3% of customers generate 39.6% of total cumulative profit** — a real,
  concentrated-value segment worth protecting/prioritizing.
- **Counter-intuitive: super customers cost *less* to acquire**, not more — avg CAC
  1,000 vs 1,452 for everyone else (0.69x). The best customers were not expensive to
  land.
- **`ad_budget` alone explains ~52% of feature importance; `budget_tier` explains 0%.**
  Since `budget_tier` is a strict tertile bucketing of `ad_budget`, once the continuous
  column is available CatBoost gets no extra information from the bucketed version —
  the engineered categorical feature turned out to be redundant, a useful negative
  result. Lower/mid ad-budget campaigns produce disproportionately more super
  customers than high-budget ones (the earlier 0%-in-High-tier finding).
- **Fewer follow-up touches, not more, associate with becoming a super customer**
  (`followup_1..5` all lower on average for super customers). This echoes the Package 2
  finding that fewer `calls_to_closed` predicts higher LTV — the same "needs less
  convincing" signal showing up a second time with a different target. Worth watching
  for a third time in Package 5 (the follow-up-call paradox package).
- **Practical answer to "how to spot them earlier":** favor leads from lower/mid
  ad-budget campaigns, and don't treat heavy follow-up activity as a positive signal —
  customers who convert with fewer touches are the ones more likely to become super
  customers later.